# 03.01 — Fine-Tune DistilBERT untuk Sentiment Bahasa (IndoNLU SMSA)

**Tujuan**: full fine-tune model encoder kecil di dataset sentiment Bahasa. Tujuan **bukan** jadi expert fine-tuning — tujuannya: feel workflow end-to-end di CPU, dan punya model siap-pakai untuk modul 04.

**Prasyarat**: modul 02 lulus.

**Model**: `distilbert-base-multilingual-cased` (134M params, mendukung 104 bahasa termasuk Indonesia).

**Dataset**: `indonlp/indonlu` config `smsa` (sentiment analysis 3 kelas: positif/netral/negatif).

**Estimasi waktu training**:
- Colab GPU: ~3-5 menit
- Colab CPU / laptop CPU: ~15-30 menit (kita subsample dataset)

Cross-link: untuk LoRA / PEFT mendalam, lihat [`llm-internals/05`](../../llm-internals/).

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Load dataset IndoNLU SMSA

In [ ]:
from datasets import load_dataset

ds = load_dataset("indonlp/indonlu", "smsa")
print(ds)
print(f"\nLabel names: {ds['train'].features['label'].names}")
print(f"Contoh:")
for i in range(3):
    ex = ds["train"][i]
    print(f"  [{ds['train'].features['label'].names[ex['label']]}] {ex['text']}")

## 2. Subsample untuk budget CPU

Dataset penuh ada ~11k train. Untuk CPU kita pakai 2000 train + 500 validation. Hasilnya tetap reasonable accuracy.

In [ ]:
SEED = 42
train_ds = ds["train"].shuffle(seed=SEED).select(range(2000))
val_ds = ds["validation"].shuffle(seed=SEED).select(range(500))
test_ds = ds["test"]

label_names = ds["train"].features["label"].names
num_labels = len(label_names)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Labels ({num_labels}): {label_names}")

## 3. Load tokenizer + tokenize

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding=False)

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
test_tok = test_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
print("Tokenization selesai.")

## 4. Load model pre-trained + tempel classifier head

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=num_labels,
    id2label=id2label, label2id=label2id,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} params (~{n_params / 1e6:.0f}M)")

## 5. Setup Trainer + metrics

In [ ]:
import numpy as np
import evaluate
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    refs = eval_pred.label_ids
    return {
        "accuracy": accuracy.compute(predictions=preds, references=refs)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=refs, average="macro")["f1"],
    }

OUTPUT_DIR = str(repo_root / "models" / "finetuned" / "distilbert-smsa")
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    save_total_limit=1,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
print("Trainer siap. Lanjut ke cell training.")

## 6. Training (3 epoch)

**Estimasi**: ~15-25 menit di CPU laptop. Di Colab GPU jauh lebih cepat (~3 menit). Akan ada log loss tiap 50 step + eval metric tiap epoch.

In [ ]:
import time

t0 = time.perf_counter()
trainer.train()
elapsed_min = (time.perf_counter() - t0) / 60
print(f"\nTraining selesai dalam {elapsed_min:.1f} menit")

## 7. Evaluasi di test set

In [ ]:
test_metrics = trainer.evaluate(test_tok)
print("Hasil di test set:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<30} {v:.4f}")

## 8. Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

preds = trainer.predict(test_tok)
y_pred = preds.predictions.argmax(axis=-1)
y_true = preds.label_ids

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — DistilBERT fine-tuned di SMSA")
plt.tight_layout()
plt.show()

## 9. Inference: kalimat custom

In [ ]:
import torch

device = next(model.parameters()).device
model.eval()

def predict(text: str) -> dict:
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = torch.softmax(logits, dim=-1)
    pred_idx = int(probs.argmax())
    return {
        "text": text,
        "label": label_names[pred_idx],
        "confidence": float(probs[pred_idx]),
        "probs": {label_names[i]: round(float(probs[i]), 3) for i in range(num_labels)},
    }

sentences = [
    "Pelayanannya buruk sekali, makanan datang lama.",
    "Tempatnya nyaman dan suasananya enak.",
    "Biasa saja sih, tidak istimewa tapi juga tidak jelek.",
    "Wah ini sih level juara, recommended banget!",
    "Mahal banget untuk porsi segini.",
]
for s in sentences:
    r = predict(s)
    print(f"[{r['label']:<8} {r['confidence']:.2f}]  {s}")

## 10. Simpan model untuk modul 04

Trainer sudah auto-save best checkpoint ke `OUTPUT_DIR`. Notebook 04.01 akan load dari sini.

In [ ]:
FINAL_DIR = str(repo_root / "models" / "finetuned" / "distilbert-smsa-final")
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Model + tokenizer disimpan ke: {FINAL_DIR}")
print("\nNote: folder ini gitignored. Untuk modul 04.01, train ulang notebook ini")
print("di environment yang sama (atau push model ke HuggingFace Hub).")

## Refleksi & insight

1. **Full fine-tune model 134M di CPU itu beneran realistic** untuk dataset 2k samples 3 epoch.
2. **DistilBERT multilingual buat task Bahasa**: accuracy biasanya 75-85% di SMSA, **cukup compete dengan LLM zero-shot** untuk task spesifik ini.
3. **Inference cepat di CPU** — DistilBERT predict satu kalimat <100 ms (vs SmolLM2 generative yang 2-5 detik).
4. **Specialist beats generalist** di domain spesifik dengan data labeled. Cerita lengkapnya di notebook 04.01.
5. **Output deterministik** (classification, bukan generation) — bukan reproducibility risk dari LLM sampling.

## Latihan mandiri

1. Coba ganti `MODEL_ID` ke `cahya/bert-base-indonesian-522M` (BERT khusus Indonesian, lebih besar). Apakah accuracy naik signifikan? Berapa lipat lebih lambat training-nya?
2. Tambah epoch jadi 5. Apakah overfit? (Indikator: train loss turun tapi val accuracy stagnan/turun.)
3. Coba dataset lain di IndoNLU: `emot` (5-class emotion) atau `wrete` (entailment). Ganti `num_labels` dan training_args yang sesuai.

## Lanjut

Modul 04: bandingkan DistilBERT yang baru kamu fine-tune ini dengan LLM zero-shot — siapa menang?
[../04-case-study/01_classification_distilbert_vs_groq.ipynb](../04-case-study/01_classification_distilbert_vs_groq.ipynb)